## WaveNet Neural Network for EEG Data Classification


In [1]:
# Importing necessities 
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, Add, Activation, Multiply
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt # plotting
import numpy as np # linear algebra
import os # accessing directory structure
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import csv
import shutil
import pathlib 
import copy
import random
import itertools


import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import scipy.signal as signal
import scipy.stats as stats
import scipy.io as sio
import matplotlib.pyplot as plt


from random import sample as sm

import glob
import os
import mat73

import os
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

import shutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

import os
import numpy as np
from scipy.io import loadmat

import tqdm
from tqdm import tqdm
import time

import os
import time
import numpy as np
import h5py
import tensorflow as tf
import json

import pickle
import tensorflow_addons as tfa

import os
import time
import h5py
import mat73
import scipy.io as sio
import random
import cupy as cp  # GPU-accelerated NumPy alternative
import numpy as np
from concurrent.futures import ProcessPoolExecutor


## Process each file and save data to HDF5 #### GOOD VERSION 20/02/2025!!!


In [2]:
import random
import time
import h5py
import scipy.io as sio
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor  # Using ThreadPoolExecutor for CPU-based parallelism
from tqdm import tqdm  # Import tqdm for progress bar

def process_file(file_info):
    """ Load .mat file and transfer data to CPU """

    file, label, base_dir = file_info
    data_path = os.path.join(base_dir, str(label), file)

    if not file.endswith('.mat'):
        return None  # Skip non-MAT files

    data = None  # Initialize data

    # Try loading with mat73 (for MATLAB 7.3)
    try:
        mat_file = mat73.loadmat(data_path)
        if 'data' in mat_file:
            data = mat_file['data'].T  # Adjust depending on structure
    except:
        pass  # No print, just try next method

    # Try loading with h5py (for MATLAB 7.3 HDF5 format)
    if data is None:
        try:
            with h5py.File(data_path, 'r') as hdf5:
                if 'data' in hdf5:
                    data = np.array(hdf5['data'])
        except:
            pass  # Try next method

    # Try loading with scipy (for older MATLAB formats)
    if data is None:
        try:
            mat_file = sio.loadmat(data_path)
            if 'data' in mat_file:
                data = mat_file['data'].flatten()
        except:
            pass  # All methods failed

    if data is None:
        return None  # Skip file if all methods failed

    # No GPU involved now, just return data on CPU
    return file, label, data  # Data is in NumPy array (CPU)

def create_hdf5_parallel(dataset_dir, output_file):
    """ Process EEG files in parallel and save to an HDF5 file """

    if os.path.exists(output_file):
        os.remove(output_file)

    with h5py.File(output_file, 'w') as hdf5_file:
        all_data = []  

        file_list = []

        # Collect all files
        for label in range(4):
            label_dir = os.path.join(dataset_dir, str(label))

            if not os.path.exists(label_dir):
                continue  # Skip missing directories

            files = os.listdir(label_dir)
            random.shuffle(files)  # Shuffle files beforehand

            for file in files:
                file_list.append((file, label, dataset_dir))

        if not file_list:
            print("No files found to process.")
            return

        # Parallel processing with a progress bar
        with ThreadPoolExecutor() as executor:
            results = list(tqdm(executor.map(process_file, file_list), total=len(file_list), desc="Processing Files"))

        # Filter out None results (failed file loads)
        results = [r for r in results if r and isinstance(r, tuple)]

        if not results:
            print("No valid EEG data was processed. Check file paths and formats.")
            return  # Avoid writing an empty HDF5 file

        # Write data into HDF5
        for filename, label, data in results:
            group = hdf5_file.require_group(f'label_{label}')  # Use label directly
            group.create_dataset(f'file_{filename}', data=data)

        print(f"Successfully saved {len(results)} EEG files to {output_file}")

# Set up the directories and timing
start_time = time.time()
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'

output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest'
dir_path = os.path.join(dataset_dir, 'DataSetTest')

#output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\Dataset_Joined'
#dir_path = os.path.join(dataset_dir, 'Dataset_Joined')
#dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
#output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\Dataset_Joined'
#dir_path = os.path.join(dataset_dir, 'Dataset_Joined')
#dir_path = os.path.join(dataset_dir, 'DatasetFnusa\\DatasetFnusa')
#output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa'
#output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest'
#dir_path = os.path.join(dataset_dir, 'DataSetTest')


# Define output HDF5 file
output_file = os.path.join(output_dir, 'dataset.h5')

# Create HDF5 file with parallel processing
create_hdf5_parallel(dir_path, output_file)

end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")


Processing Files: 100%|██████████| 575/575 [00:11<00:00, 52.20it/s]


Successfully saved 575 EEG files to C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\dataset.h5
Elapsed time: 12.731290578842163 seconds


In [4]:
import os
import time
import numpy as np
import h5py
import mat73
import scipy.io as sio
import random  # Added for shuffling

# ---------------------------
# 1. Set Up Directories & Timing
# ---------------------------
start_time = time.time()
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
#output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\Dataset_Joined'
#dir_path = os.path.join(dataset_dir, 'Dataset_Joined')
#dir_path = os.path.join(dataset_dir, 'DatasetFnusa\\DatasetFnusa')
#output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa'
output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest'
dir_path = os.path.join(dataset_dir, 'DataSetTest')

# Define output HDF5 file
output_file = os.path.join(output_dir, 'dataset.h5')

# ---------------------------
# 2. Function to Process and Shuffle Data
# ---------------------------
def process_file(file, label, hdf5_file, base_dir):
    if file.endswith('.mat'):
        data_path = os.path.join(base_dir, str(label), file)
        try:
            mat_file = mat73.loadmat(data_path)
            data = mat_file['data'].T
        except:
            mat_file = sio.loadmat(data_path)
            data = mat_file['data'].flatten()
        return file, data  # Return both filename and data

def create_hdf5(dataset_dir, output_file):
    # Remove existing HDF5 file to avoid issues
    if os.path.exists(output_file):
        os.remove(output_file)

    with h5py.File(output_file, 'w') as hdf5_file:
        all_data = []  # Store (label, filename, data) tuples

        # Load and shuffle files before storing
        for label in range(4):
            label_dir = os.path.join(dataset_dir, str(label))
            files = os.listdir(label_dir)
            random.shuffle(files)  # **Shuffle file order**
            
            for file in files:
                result = process_file(file, label, hdf5_file, dataset_dir)
                if result:
                    filename, data = result
                    all_data.append((label, filename, data))

        # Shuffle all collected data before writing
        random.shuffle(all_data)

        # Write shuffled data into HDF5
        for label, filename, data in all_data:
            hdf5_file.create_dataset(f'label_{label}/file_{filename}', data=data)

# ---------------------------
# 3. Create the HDF5 File
# ---------------------------
create_hdf5(dir_path, output_file)

end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")


Elapsed time: 1.7233328819274902 seconds


## Check if random

In [5]:
with h5py.File(output_file, 'r') as hdf5_file:
    for label in range(4):
        print(f"Label {label} samples:", list(hdf5_file[f'label_{label}'].keys())[:10])  # Print sample filenames


Label 0 samples: ['file_p000005281.mat', 'file_p000005282.mat', 'file_p000005283.mat', 'file_p000005284.mat', 'file_p000005285.mat', 'file_p000005286.mat', 'file_p000010219 (2).mat', 'file_p000010219.mat', 'file_p000010220 (2).mat', 'file_p000010220.mat']
Label 1 samples: ['file_p000016272.mat', 'file_p000016273.mat', 'file_p000016274.mat', 'file_p000016275.mat', 'file_p000016276.mat', 'file_p000016277.mat', 'file_p000016278.mat', 'file_p000016279.mat', 'file_p000029021.mat', 'file_p000029022.mat']
Label 2 samples: ['file_p000048646.mat', 'file_p000048647.mat', 'file_p000048648.mat', 'file_p000048649.mat', 'file_p000048650.mat', 'file_p000048651.mat', 'file_p000048652.mat', 'file_p000048653.mat', 'file_p000048654.mat', 'file_p000058735 (2).mat']
Label 3 samples: ['file_p000092070.mat', 'file_p000092071.mat', 'file_p000092072.mat', 'file_p000092073.mat', 'file_p000092074.mat', 'file_p000092075.mat', 'file_p000092076.mat', 'file_p000092077.mat', 'file_p000092078.mat', 'file_p000103072.ma

## Split and save HDF5 data

In [6]:
import os
import time
import json
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, Callback
import random

# ---------------------------
# 1. Enable GPU Memory Growth (Fix GPU Memory Allocation Issue)
# ---------------------------
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# ---------------------------
# 2. Set Up Directories & Paths
# ---------------------------
start_time = time.time()
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest'

output_file = os.path.join(output_dir, 'dataset.h5')
train_file = os.path.join(output_dir, 'train.h5')
val_file = os.path.join(output_dir, 'val.h5')
test_file = os.path.join(output_dir, 'test.h5')

# ---------------------------
# 3. Define HDF5 Generator Function (Fix Integer Label Conversion)
# ---------------------------
def hdf5_generator(file_path):
    """Generator function to load EEG data from HDF5 with proper type handling."""
    with h5py.File(file_path, 'r') as hdf5_file:
        all_files = []

        for label in hdf5_file.keys():
            label = label.decode() if isinstance(label, bytes) else label  # Ensure string label
            for file_name in hdf5_file[label].keys():
                file_name = file_name.decode() if isinstance(file_name, bytes) else file_name  # Ensure string filename
                all_files.append((label, file_name))

        np.random.shuffle(all_files)  # Shuffle data

    for label, file_name in all_files:
        with h5py.File(file_path, 'r') as hdf5_file:
            dataset_key = f"{label}/{file_name}"  # Ensure correct formatting
            if dataset_key not in hdf5_file:
                print(f"Warning: {dataset_key} not found in {file_path}")
                continue  # Skip missing files

            data = hdf5_file[dataset_key][:]
            data = data.astype(np.float32)
            data = (data - np.mean(data)) / (np.std(data) + 1e-8)  # Normalize

            int_label = int(label.split('_')[-1])  # Convert "label_0" → 0
            # Fix: Convert label and data to TensorFlow tensors
            yield tf.convert_to_tensor(data.reshape((1, 1, 15000)), dtype=tf.float32), \
                  tf.convert_to_tensor(np.array([[int_label]]), dtype=tf.int64)

# ---------------------------
# 4. Create TensorFlow Dataset (Fix Output Signature)
# ---------------------------
def create_tf_dataset(file_path, batch_size, shuffle=True):
    """Creates a TensorFlow dataset from the HDF5 generator with optional shuffling."""
    dataset = tf.data.Dataset.from_generator(
        lambda: hdf5_generator(file_path),
        output_signature=(
            tf.TensorSpec(shape=(1, 1, 15000), dtype=tf.float32),
            tf.TensorSpec(shape=(1, 1), dtype=tf.int64)
        )
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=50000, reshuffle_each_iteration=True)  # Ensures each epoch is shuffled
    return dataset.batch(batch_size).repeat()

# ---------------------------
# 5. Split and Save Data
# ---------------------------
def save_to_hdf5(file_path, data_list):
    if os.path.exists(file_path):
        os.remove(file_path)  # Remove existing file
    with h5py.File(file_path, 'w') as hdf5_file:
        for label, fname, data in data_list:
            group = hdf5_file.require_group(f"label_{label}")
            group.create_dataset(fname, data=data)

def split_and_save_data(file_path, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    all_data = []
    
    with h5py.File(file_path, 'r') as hdf5_file:
        for label in hdf5_file.keys():
            for fname in hdf5_file[label]:
                data = hdf5_file[f"{label}/{fname}"][:]
                all_data.append((label, fname, data))

    num_total = len(all_data)
    num_train = int(train_ratio * num_total)
    num_val = int(val_ratio * num_total)
    num_test = num_total - num_train - num_val  # Ensures all data is used

    save_to_hdf5(train_file, all_data[:num_train])
    save_to_hdf5(val_file, all_data[num_train:num_train + num_val])
    save_to_hdf5(test_file, all_data[num_train + num_val:])  # Remaining 10%

# Call the function with correct split
split_and_save_data(output_file, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1)

    

# ---------------------------
# 6. Estimate Dataset Sizes & Steps
# ---------------------------
def count_samples(file_path):
    """Count total samples in an HDF5 file across all labels."""
    with h5py.File(file_path, 'r') as hdf5_file:
        return sum(len(hdf5_file[label]) for label in hdf5_file)

# Fix: Reduce batch size to avoid memory overflow
batch_size = 32  # Adjusted from 128 to 32

# Fix: Add CPU Fallback if GPU still causes issues
try:
    with tf.device('/GPU:0'):
        train_dataset = create_tf_dataset(train_file, batch_size, shuffle=True)
        val_dataset = create_tf_dataset(val_file, batch_size, shuffle=True)
        test_dataset = create_tf_dataset(test_file, batch_size, shuffle=True)
except:
    print("GPU allocation failed, switching to CPU...")
    with tf.device('/CPU:0'):
        train_dataset = create_tf_dataset(train_file, batch_size, shuffle=True)
        val_dataset = create_tf_dataset(val_file, batch_size, shuffle=True)
        test_dataset = create_tf_dataset(test_file, batch_size, shuffle=True)

total_train_samples = count_samples(train_file)
total_val_samples = count_samples(val_file)
total_test_samples = count_samples(test_file)

steps_per_epoch = max(total_train_samples // batch_size, 1)
validation_steps = max(total_val_samples // batch_size, 1)
test_steps = max(total_test_samples // batch_size, 1)

print(f"Train samples: {total_train_samples}, Steps per epoch: {steps_per_epoch}")
print(f"Validation samples: {total_val_samples}, Validation steps: {validation_steps}")
print(f"Test samples: {total_test_samples}, Test steps: {test_steps}")

end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")


Train samples: 402, Steps per epoch: 12
Validation samples: 115, Validation steps: 3
Test samples: 58, Test steps: 1
Elapsed time: 1.3597681522369385 seconds


In [7]:
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Compile, train and save the model

In [8]:
import tensorflow as tf
import os
import json
import time
from tensorflow.keras.callbacks import EarlyStopping

# ---------------------------
# 1. Enable GPU Usage and Optimizations
# ---------------------------
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevent full memory allocation
        print(f"Using GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found, using CPU.")

# Enable XLA Compilation for faster execution
#tf.config.optimizer.set_jit(True)

# Use mixed precision for better GPU performance
tf.keras.mixed_precision.set_global_policy('mixed_float16')

start_time = time.time()

# ---------------------------
# 2. Define Swish Activation Function
# ---------------------------
def swish_activation(x):
    return x * tf.sigmoid(x)

# ---------------------------
# 3. Define and Compile WaveNet Model
# ---------------------------
wavenet_model = tf.keras.Sequential([
    tf.keras.layers.BatchNormalization()
])

l2_reg = 0.001  # Reduced for efficiency

for rate in [1, 2, 4, 8, 16, 32, 64]:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation=swish_activation,
        dilation_rate=rate,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
    ))
    wavenet_model.add(tf.keras.layers.Dropout(0.1))  # Increased for better generalization

wavenet_model.add(tf.keras.layers.Conv1D(filters=4, kernel_size=1, kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
wavenet_model.add(tf.keras.layers.Activation('softmax', dtype='float32'))  # Ensure final output in float32

# Compile with AdamW Optimizer for speed & stability
wavenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------------------
# 4. Define Callbacks
# ---------------------------
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,  
    restore_best_weights=True
)

# ---------------------------
# 5. Train the Model with GPU Acceleration
# ---------------------------

def train_model():
    return wavenet_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=150,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        callbacks=[early_stopping]
    )

history = train_model()

# ---------------------------
# 6. Evaluate & Save Model
# ---------------------------
test_loss, test_accuracy = wavenet_model.evaluate(test_dataset, steps=test_steps)
print(f"Test accuracy: {test_accuracy:.4f}")

# Save model
model_save_path = os.path.join(output_dir, 'Wavenet_model_Test')
wavenet_model.save(model_save_path)
print(f"Model saved to {model_save_path}")

# Save training history
with open(os.path.join(output_dir, 'training_history.json'), 'w') as f:
    json.dump(history.history, f)

# Print elapsed time
print(f"Elapsed time: {time.time() - start_time:.2f} seconds")


Using GPU: /physical_device:GPU:0
Epoch 1/150
12/12 [==============================] - 12s 187ms/step - loss: 1.5265 - accuracy: 0.4766 - val_loss: 1.7053 - val_accuracy: 0.3750
Epoch 2/150
12/12 [==============================] - 2s 169ms/step - loss: 1.3190 - accuracy: 0.5162 - val_loss: 2.0576 - val_accuracy: 0.3854
Epoch 3/150
12/12 [==============================] - 2s 171ms/step - loss: 1.2431 - accuracy: 0.5162 - val_loss: 2.3044 - val_accuracy: 0.3854
Epoch 4/150
1/1 [==============================] - 0s 198ms/step - loss: 2.4031 - accuracy: 0.0000e+00
Test accuracy: 0.0000


INFO:tensorflow:Assets written to: C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\Wavenet_model_Test\assets


INFO:tensorflow:Assets written to: C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\Wavenet_model_Test\assets


Model saved to C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\Wavenet_model_Test
Elapsed time: 21.47 seconds


In [9]:
# Plot training & validation accuracy values
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.savefig(os.path.join(output_dir, 'Model_acc_val_loss.jpeg'))
plt.show()

# Plot per-step loss values
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(step_history.step_losses)
plt.title('Per-Step Loss')
plt.ylabel('Loss')
plt.xlabel('Step')

# Plot per-step accuracy values
plt.subplot(1, 2, 2)
plt.plot(step_history.step_acc)
plt.title('Per-Step Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Step')

plt.savefig(os.path.join(output_dir, 'step_loss_acc.jpeg'))
plt.show()

# Plot per-step validation loss values
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(step_history.val_step_losses)
plt.title('Per-Epoch Validation Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')


# Plot per-step validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(step_history.val_step_acc)
plt.title('Per-Epoch Validation Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')

# Save plots
plt.savefig(os.path.join(output_dir, 'epoch_val_ac.jpeg'))
plt.show()

NameError: name 'plt' is not defined

In [52]:
#Test with JOINED DATASET
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa\Output'
# Define the directory containing your dataset
dir = os.path.join(dataset_dir, 'DatasetFnusa\\DatasetFnusa')
model_path = os.path.join(output_dir, 'Wavenet_model_Test')
wavenet_model = tf.keras.models.load_model(model_path, custom_objects={'swish_activation': swish_activation})  # Load the model
start_time = time.time()
output_file = os.path.join(output_dir, 'dataset.h5')
train_files = [os.path.join(output_dir, f'train_label_{label}.h5') for label in labels]
val_files = [os.path.join(output_dir, f'val_label_{label}.h5') for label in labels]
test_files = [os.path.join(output_dir, f'test_label_{label}.h5') for label in labels]


# Calculate total samples for each dataset
total_train_samples = sum(count_samples_model(file) for file in train_files)
total_val_samples = sum(count_samples_model(file) for file in val_files)
total_test_samples = sum(count_samples_model(file) for file in test_files)

# Calculate steps per epoch
steps_per_epoch = int(total_train_samples // batch_size)
validation_steps = int(total_val_samples // batch_size)
test_steps = int(total_test_samples // batch_size)

# Ensure the steps are at least 1
steps_per_epoch = max(steps_per_epoch, 1)
validation_steps = max(validation_steps, 1)
test_steps = max(test_steps, 1)

# Define Swish activation function
def swish_activation(x):
    return x * tf.sigmoid(x)
def get_true_labels_and_predictions(wavenet_model, dataset, steps):
    true_labels = []
    predictions = []

    for step, (batch_data, batch_labels) in enumerate(dataset.take(steps)):
        print("batch_data shape:", batch_data.shape)
        print("batch_data dtype:", batch_data.dtype)
        print("batch_labels shape:", batch_labels.shape)
        print("batch_labels dtype:", batch_labels.dtype)

        preds = model.predict(batch_data)
        print("First few predictions:\n", preds[:5])

        preds = np.argmax(preds, axis=-1).flatten()
        true_labels.extend(batch_labels.numpy().flatten())
        predictions.extend(preds)

    return np.array(true_labels), np.array(predictions)

test_loss, test_accuracy = wavenet_model.evaluate(test_dataset, steps=test_steps)
print(f"Test accuracy: {test_accuracy}")
true_labels, predictions = get_true_labels_and_predictions(wavenet_model, test_dataset, test_steps)

report = classification_report(true_labels, predictions, output_dict=True, zero_division=1) # Handle zero division

FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = 'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa\Output\train_label_0.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

# Generate spikes



In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def generate_eeg_waveform(model, seed, num_generated, seed_length):
    """
    Autoregressively generate a sequence from the WaveNet model for EEG waveform generation.
    
    Parameters:
        model         : The trained WaveNet model.
        seed          : Initial seed as a 1D numpy array of quantized values (e.g., integers 0-3).
        num_generated : Number of new samples to generate.
        seed_length   : The length of the seed (and the fixed window for the model, if needed).
        
    Returns:
        generated_seq : A numpy array containing the seed followed by generated quantized values.
    """
    generated_seq = list(seed)  # Start with the seed sequence
    current_input = np.array(seed, dtype=np.float32)[None, :]  # Shape: (1, seed_length)
    
    for _ in range(num_generated):
        # Predict probabilities for the next sample (output shape: (1, seq_length, num_classes))
        predictions = model.predict(current_input)
        next_prob = predictions[0, -1, :]  # Get the probability distribution for the last timestep
        
        # Sample the next quantized value based on the predicted distribution
        next_sample = np.random.choice(len(next_prob), p=next_prob)
        generated_seq.append(next_sample)
        
        # Update the input window (maintaining the fixed seed_length)
        current_input = np.concatenate(
            [current_input, np.array([[next_sample]], dtype=np.float32)], axis=1
        )
        if current_input.shape[1] > seed_length:
            current_input = current_input[:, -seed_length:]
    
    return np.array(generated_seq)

# ---------------------------
# Example Usage
# ---------------------------
seed_length = 100  # Define a window of 100 samples
seed = np.random.randint(0, 4, size=seed_length)  # Create an initial seed with quantized values (0-3)
num_generated = 1000  # Number of additional samples to generate

# Generate a sequence of quantized values from the WaveNet model
generated_sequence = generate_eeg_waveform(wavenet_model, seed, num_generated, seed_length)

# ---------------------------
# Postprocessing: Map Quantized Values to EEG Amplitudes
# ---------------------------
# For example, mapping quantized values to microvolt ranges: 
# 0 -> -100 µV, 1 -> -33 µV, 2 -> 33 µV, 3 -> 100 µV.
amplitude_mapping = {0: -100, 1: -33, 2: 33, 3: 100}
eeg_waveform = np.array([amplitude_mapping[val] for val in generated_sequence], dtype=np.float32)

# ---------------------------
# Visualize the Generated EEG Waveform
# ---------------------------
plt.figure(figsize=(12, 4))
plt.plot(eeg_waveform)
plt.title("Generated EEG Waveform")
plt.xlabel("Time (sample index)")
plt.ylabel("Amplitude (µV)")
plt.show()


In [9]:
# Calculate labels per category (true labels)
true_labels_per_category = np.sum(cm, axis=1)

# Print the true labels per category
print("\nTrue Labels per Category:")
print(true_labels_per_category)



NameError: name 'cm' is not defined

## TRY 3 09/04/2025

In [13]:
import os
import numpy as np
import mat73
import scipy.io

def convert_mat_to_npy(mat_file_path, npy_file_path):
    """Converts a .mat file to a .npy file."""
    try:
        mat_data = mat73.loadmat(mat_file_path)
        data = mat_data['data'].T
    except TypeError:
        mat_data = scipy.io.loadmat(mat_file_path)
        data = mat_data['data'].flatten()
    except OSError:
        mat_data = scipy.io.loadmat(mat_file_path)
        data = mat_data['data'].flatten()

    np.save(npy_file_path, data)

data_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa\DatasetFnusa'

for label in range(4):
    label_dir = os.path.join(data_dir, str(label))
    for file in os.listdir(label_dir):
        if file.endswith('.mat'):
            mat_file_path = os.path.join(label_dir, file)
            npy_file_path = os.path.join(label_dir, file.replace('.mat', '.npy'))
            convert_mat_to_npy(mat_file_path, npy_file_path)


import tensorflow as tf
import os
import numpy as np

def load_and_process(file_path, label):
    """Loads and processes a single .npy file."""
    data = np.load(file_path)
    data = data.astype(np.float32)
    data = (data - np.mean(data)) / (np.std(data) + 1e-8)  # Normalize
    return data.reshape((1, 1, 15000)), np.array([[label]], dtype=np.int64)

def create_dataset(data_dir, batch_size):
    """Creates a TensorFlow dataset directly from .npy files."""
    file_paths = []
    labels = []

    for label in range(4):
        label_dir = os.path.join(data_dir, str(label))
        for file in os.listdir(label_dir):
            if file.endswith('.npy'):
                file_paths.append(os.path.join(label_dir, file))
                labels.append(label)

    def generator():
        for file_path, label in zip(file_paths, labels):
            yield load_and_process(file_path, label)

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(1, 1, 15000), dtype=tf.float32),
            tf.TensorSpec(shape=(1, 1), dtype=tf.int64)
        )
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Example Usage
data_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa\DatasetFnusa'
batch_size = 32
dataset = create_dataset(data_dir, batch_size)

# Split dataset into train, validation, and test.
dataset_size = len(list(dataset))
train_size = int(0.7 * dataset_size)
val_size = int(0.2 * dataset_size)

train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size).take(val_size)
test_dataset = dataset.skip(train_size + val_size)

steps_per_epoch = train_size // batch_size
validation_steps = val_size // batch_size
test_steps = (dataset_size - train_size - val_size) // batch_size

KeyboardInterrupt: 

In [14]:
import numpy as np
import os
import mat73
import scipy.io

def load_mat_data(file_path):
    """Loads .mat data into a NumPy array."""
    try:
        mat_file = mat73.loadmat(file_path)
        data = mat_file['data'].T
    except TypeError:
        mat_file = scipy.io.loadmat(file_path)
        data = mat_file['data'].flatten()
    except OSError:
        mat_file = scipy.io.loadmat(file_path)
        data = mat_file['data'].flatten()
    return data

def load_data_into_lists(data_dir):
    """Loads .mat data into lists."""
    data_list = []
    label_list = []

    for label in range(4):
        label_dir = os.path.join(data_dir, str(label))
        for file in os.listdir(label_dir):
            if file.endswith('.mat'):
                file_path = os.path.join(label_dir, file)
                data = load_mat_data(file_path)
                data_list.append(data)
                label_list.append(label)

    return np.array(data_list), np.array(label_list)

# Example Usage
data_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DatasetFnusa\DatasetFnusa'
data, labels = load_data_into_lists(data_dir)

# Now you can use 'data' and 'labels' with your Keras model
# For example, you can create a TensorFlow dataset from these lists:
import tensorflow as tf
dataset = tf.data.Dataset.from_tensor_slices((data, labels))
#And then batch, and prefetch the dataset.
dataset = dataset.batch(32).prefetch(tf.data.AUTOTUNE)

#Then use this dataset in your keras model.

KeyboardInterrupt: 

In [8]:
import tensorflow as tf
import os
import json
import time
from tensorflow.keras.callbacks import EarlyStopping

# ---------------------------
# 1. Enable GPU Usage and Optimizations
# ---------------------------
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(f"GPU allocation failed: {e}") #More detailed error message
else:
    print("No GPU found, using CPU.")

# Enable XLA Compilation for faster execution
tf.config.optimizer.set_jit(True)

# Use mixed precision for better GPU performance
tf.keras.mixed_precision.set_global_policy('mixed_float16')

start_time = time.time()

# ---------------------------
# 2. Define Swish Activation Function
# ---------------------------
def swish_activation(x):
    return x * tf.sigmoid(x)

# ---------------------------
# 3. Define and Compile WaveNet Model
# ---------------------------
wavenet_model = tf.keras.Sequential([
    tf.keras.layers.BatchNormalization()
])

l2_reg = 0.001  # Reduced for efficiency

for rate in [1, 2, 4, 8, 16, 32, 64]:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation=swish_activation,
        dilation_rate=rate,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)
    ))
    wavenet_model.add(tf.keras.layers.Dropout(0.2))  # Increased dropout

wavenet_model.add(tf.keras.layers.Conv1D(filters=4, kernel_size=1, kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
wavenet_model.add(tf.keras.layers.Activation('softmax', dtype='float32'))

# Compile with AdamW Optimizer for speed & stability
wavenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------------------
# 4. Define Callbacks
# ---------------------------
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# ---------------------------
# 5. Train the Model with GPU Acceleration
# ---------------------------

def train_model():
    return wavenet_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=150,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        callbacks=[early_stopping]
    )

history = train_model()

# ---------------------------
# 6. Evaluate & Save Model
# ---------------------------
test_loss, test_accuracy = wavenet_model.evaluate(test_dataset, steps=test_steps)
print(f"Test accuracy: {test_accuracy:.4f}")

# Save model
model_save_path = os.path.join(output_dir, 'Wavenet_model_Test')
wavenet_model.save(model_save_path)
print(f"Model saved to {model_save_path}")

# Save training history
with open(os.path.join(output_dir, 'training_history.json'), 'w') as f:
    json.dump(history.history, f)

# Print elapsed time
print(f"Elapsed time: {time.time() - start_time:.2f} seconds")

Using GPU: /physical_device:GPU:0


NameError: name 'train_dataset' is not defined

In [ ]:
import matplotlib.pyplot as plt
import os

# Assuming 'history' is from model.fit()

# Plot training & validation accuracy values
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

try:
    plt.savefig(os.path.join(output_dir, 'Model_acc_val_loss.jpeg'))
except Exception as e:
    print(f"Error saving Model_acc_val_loss plot: {e}")

plt.show()

# --- Placeholder for step_history (you'll need to define this) ---
# Example of a basic structure. You will need to make a custom callback to make this work.
class StepHistory:
    def __init__(self):
        self.step_losses = []
        self.step_acc = []
        self.val_step_losses = []
        self.val_step_acc = []

# step_history = StepHistory()
# -----------------------------------------------------------------

if 'step_losses' in vars() or 'step_losses' in globals(): #check if step_history is defined.
    # Plot per-step loss values
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(step_history.step_losses)
    plt.title('Per-Step Loss')
    plt.ylabel('Loss')
    plt.xlabel('Step')

    # Plot per-step accuracy values
    plt.subplot(1, 2, 2)
    plt.plot(step_history.step_acc)
    plt.title('Per-Step Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Step')

    try:
        plt.savefig(os.path.join(output_dir, 'step_loss_acc.jpeg'))
    except Exception as e:
        print(f"Error saving step_loss_acc plot: {e}")
    plt.show()

    # Plot per-step validation loss values
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(step_history.val_step_losses)
    plt.title('Per-Step Validation Loss') # changed to per step
    plt.ylabel('Loss')
    plt.xlabel('Step') # changed to per step

    # Plot per-step validation accuracy values
    plt.subplot(1, 2, 2)
    plt.plot(step_history.val_step_acc)
    plt.title('Per-Step Validation Accuracy') # changed to per step
    plt.ylabel('Accuracy')
    plt.xlabel('Step') # changed to per step

    try:
        plt.savefig(os.path.join(output_dir, 'epoch_val_ac.jpeg'))
    except Exception as e:
        print(f"Error saving epoch_val_ac plot: {e}")
    plt.show()
else:
    print("step_history is not defined. Skipping per step plots.")